In [ ]:
import numpy as np

# Grid World Enviroment
# 0 = Free, 1 = Objective, 2 = Hole, 3 = Obstacle
world = np.array([[0, 0, 0, 0, 2], 
                  [0, 3, 3, 0, 0],
                  [0, 3, 0, 0, 2],
                  [0, 0, 0, 0, 0],
                  [0, 3, 2, 1, 0],])

In [ ]:
def states(world) -> dict:
    set_of_states = {}
    labels = {0: "Free", 
              1: "Objective", 
              2: "Hole", 
              3: "Obstacle"}
    
    for row in range(world.shape[0]):
        for col in range(world.shape[1]):
            state = (row, col)
            set_of_states[state] = labels[world[row, col]]

    return set_of_states


def actions() -> dict:
    set_of_actions = {"LEFT": (0, -1),
                      "RIGHT": (0, 1), 
                      "DOWN": (1, 0), 
                      "UP": (-1, 0)}
    
    return set_of_actions

# add probability
def transitions(set_of_states, set_of_actions, deterministic=True) -> dict:
    set_of_transitions = {}
    for s1 in set_of_states.keys():
        set_of_transitions[s1] = {}

        for action, movement in set_of_actions.items():
            s2 = tuple(int(x) for x in np.array(s1) + np.array(movement))

            if not (0 <= s2[0] < world.shape[0] and 0 <= s2[1] < world.shape[1]):
                s2 = s1

            elif set_of_states[s2] == "Obstacle":
                s2 = s1

            # its necessary to have the probability "p"
            set_of_transitions[s1][action] = tuple(s2)

    return set_of_transitions


def rewards(world) -> dict:
    set_of_rewards = {}
    reward_values = {0: -1,   # Free
                     1: 10,   # Objective
                     2: -10,  # Hole
                     3: 0}    # Obstacle

    for row in range(world.shape[0]):
        for col in range(world.shape[1]):
            state = (row, col)
            set_of_rewards[state] = reward_values[world[row, col]]

    return set_of_rewards

# explotation and exploration
# finite and infinite horizon
time_step = 0 # D

In [ ]:
set_of_states = states(world)
set_of_actions = actions()
set_of_transitions = transitions(set_of_states, set_of_actions, deterministic=True)
set_of_rewards = rewards(world)

print(set_of_states)
print(set_of_actions)
print(set_of_transitions)
print(set_of_rewards)

### Deterministic

In [ ]:
# Convert into a class
mdp = {"S": states(world),
       "A": actions(),
       "T": transitions(set_of_states, set_of_actions, deterministic=True),
       "R": rewards(world),
       "gamma": 0.9}

In [ ]:
def value_iteration(states, actions, transitions, rewards,
                    gamma=0.9, theta=1e-4):

    V = {state: 0 for state in states}

    while True:
        delta = 0
        new_V = V.copy()

        for state in states:

            # Ignorar obstáculos
            if states[state] == "Obstacle":
                continue

            # Estados terminais não recebem valor futuro
            if states[state] in ("Objective", "Hole"):
                continue

            values = []

            for action in actions:
                next_state = transitions[state][action]
                value = rewards[next_state] + gamma * V[next_state]
                values.append(value)

            new_V[state] = max(values)

            delta = max(
                delta,
                abs(new_V[state] - V[state])
            )

        V = new_V

        if delta < theta:
            break

    return V

V = value_iteration(set_of_states, set_of_actions, set_of_transitions, set_of_rewards)

for state, value in V.items():
    print(state, round(value, 2))


In [ ]:
value_matrix = np.zeros(world.shape)

for (row, col), value in V.items():
    value_matrix[row, col] = value

print(np.round(value_matrix, 4))